In [1]:
import sys
sys.path.append("..")

In [2]:
import cvxpy as cp
import itertools
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from src.mip import solve, solve_milp

In [ ]:
pretty_template = go.layout.Template(
    layout=dict(
        font=dict(family="iosevka"), 
        margin=dict(t=50, b=30, l=30, r=30),
    )
)

pio.templates["pretty"] = pretty_template
pio.templates.default = "plotly+pretty"

In [ ]:
# M = [21]
# N = [8,9,10,11,12,13,14]
M = [21, 31, 41, 51]
N = [8,10,12,14]

df = pd.DataFrame()
for m in M:
    for n in N:
        # df_mn = pd.read_pickle(f"grid_search_milp_miqp_greedy_m{m}_n{n}.pkl")
        df_mn = pd.read_pickle(f"grid_search_solver_greedy_m{m}_n{n}.pkl")
        df_mn["m"] = m
        df_mn["n"] = n
        df = pd.concat((df, df_mn))

df = df.sort_values(["m", "n", "i", "alg"]).reset_index(drop=True)

In [ ]:
df.head(10)

In [ ]:
df_agg = df.groupby(["m", "n", "alg"], as_index=False).mean(True)
df_agg.head()

In [ ]:
# fig = px.line(df_agg[df_agg["alg"]=="MILP_R"], x="m", y="time", facet_col="n", labels={"time": "time (s)", "n": "n_classifiers"}, markers=True, title="LP Solve Time")
fig = px.line(df_agg, x="m", y="time", facet_col="n", color="alg", labels={"time": "time (s)", "n": "n_classifiers"}, markers=True, title="LP Solve Time")
fig.update_layout(title=dict(xanchor="center", x= 0.5))

In [ ]:
fig = px.line(df_agg, x="m", y="loss", color="alg", facet_col="n", labels={"time": "time (s)", "n": "n_classifiers"}, markers=True, title="LP vs Greedy Loss")
fig.update_layout(title=dict(xanchor="center", x= 0.5))

In [ ]:
if True:
    M = [21, 31, 41]
    N = [8,10,12,14]

    df_all = pd.DataFrame()
    for m in M:
        for n in N:
            try:
                df_mn = pd.read_pickle(f"grid_search_milp_miqp_greedy_m{m}_n{n}.pkl")
                df_mn = df_mn[df_mn["alg"]=="MIQP_R"]
                df_mn["m"] = m
                df_mn["n"] = n
                df_all = pd.concat((df_all, df_mn))
            except:
                pass

    df_all = df_all.sort_values(["m", "n", "i", "alg"]).reset_index(drop=True)
    df4 = pd.concat((df, df_all[df.columns]))
    df4_agg = df4.groupby(["m", "n", "alg"], as_index=False).mean(True)

In [ ]:
fig = px.line(df4_agg[(df4_agg["alg"]=="MILP_R") | (df4_agg["alg"]=="MIQP_R")], color="alg", x="m", y="time", facet_col="n", labels={"time": "time (s)", "n": "n_classifiers"}, markers=True, title="LP & QP Solve Time")
fig.update_layout(title=dict(xanchor="center", x= 0.5))

### Extra Analysis

In [3]:
M = [11, 21]
N = [4, 5, 6]
df_cp = pd.DataFrame()
for m in M:
    for n in N:
        # df_mn = pd.read_pickle(f"grid_search_milp_miqp_greedy_m{m}_n{n}.pkl")
        df_mn = pd.read_pickle(f"grid_search_solver_m{m}_n{n}.pkl")
        df_mn["m"] = m
        df_mn["n"] = n
        df_cp = pd.concat((df_cp, df_mn))

df_cp = df_cp.sort_values(["m", "n", "i", "alg"]).reset_index(drop=True)

In [4]:
df_cp

,i,alg,time,loss,partition,m,n
0,0,brute force,0.001121,0.081818,"[[0], [1], [2, 3]]",11,4
1,0,pruned,0.056114,0.086364,"[[0], [1, 2], [3]]",11,4
2,1,brute force,0.001274,0.081818,"[[0], [1], [2, 3]]",11,4
3,1,pruned,0.055469,0.086364,"[[0], [1], [2], [3]]",11,4
4,2,brute force,0.001049,0.086364,"[[0], [1, 2], [3]]",11,4
...,...,...,...,...,...,...,...
65887,11625,pruned,0.398413,0.004762,"[[0, 1, 2, 3, 4, 5]]",21,6
65888,11626,brute force,0.017504,0.004762,"[[0, 1, 2, 3, 4, 5]]",21,6
65889,11626,pruned,0.278044,0.004762,"[[0, 1, 2, 3, 4, 5]]",21,6
65890,11627,brute force,0.017745,0.004762,"[[0, 1, 2, 3, 4, 5]]",21,6


In [5]:
res_ratio = {"m": [], "n": [], "i": [], "alg": [], "ratio": [], "time": []}
# res_ratio = {"m": [], "n": [], "i": [], "alg": [], "time": []}
for m in M:
    for n in N:
        df_mn = df_cp[(df_cp["m"]==m) & (df_cp["n"]==n)]
        for alg in ["brute force", "pruned"]:
            r = (df_mn[df_mn["alg"]==alg]["loss"].reset_index(drop=True) / df_mn[df_mn["alg"]=="brute force"]["loss"].reset_index(drop=True))
            t = df_mn[df_mn["alg"]==alg]["time"]
            i = df_mn[df_mn["alg"]==alg]["i"]

            res_ratio["m"].extend([m]*len(t))
            res_ratio["n"].extend([n]*len(t))
            res_ratio["i"].extend(i)
            res_ratio["alg"].extend([alg]*len(t))
            res_ratio["ratio"].extend(r.tolist())
            res_ratio["time"].extend(t.tolist())

        # r_gdy = np.minimum(df_mn[df_mn["alg"]=="AGG"].loss.to_numpy(), df_mn[df_mn["alg"]=="DIV"].loss.to_numpy()) / df_mn[df_mn["alg"]=="OPT"]["loss"].reset_index(drop=True)
        # t_gdy = (df_mn[df_mn["alg"]=="AGG"].time.to_numpy() + df_mn[df_mn["alg"]=="DIV"].time.to_numpy())
        # i_gdy = df_mn[df_mn["alg"]=="AGG"]["i"]
        
        # res_ratio["m"].extend([m]*len(r_gdy))
        # res_ratio["n"].extend([n]*len(r_gdy))
        # res_ratio["i"].extend(i_gdy)
        # res_ratio["alg"].extend(["GDY"]*len(r_gdy))
        # res_ratio["ratio"].extend(r_gdy.tolist())
        # res_ratio["time"].extend(t_gdy.tolist())

In [6]:
df_res_ratio = pd.DataFrame(res_ratio)
df_res_ratio

,m,n,i,alg,ratio,time
0,11,4,0,brute force,1.0,0.001121
1,11,4,1,brute force,1.0,0.001274
2,11,4,2,brute force,1.0,0.001049
3,11,4,3,brute force,1.0,0.001027
4,11,4,4,brute force,1.0,0.001011
...,...,...,...,...,...,...
65887,21,6,11623,pruned,1.0,0.280690
65888,21,6,11624,pruned,1.0,0.283579
65889,21,6,11625,pruned,1.0,0.398413
65890,21,6,11626,pruned,1.0,0.278044


In [7]:
idx = pd.IndexSlice
df_res_ratio.groupby(["alg", "m", "n"]).describe().loc[:, idx[["ratio", "time"], ["mean", "std", "min", "max"]]]

ratio                               time            \
                      mean       std  min       max      mean       std   
alg         m  n                                                          
brute force 11 4  1.000000  0.000000  1.0  1.000000  0.001019  0.000042   
               5  1.000000  0.000000  1.0  1.000000  0.004032  0.000340   
               6  1.000000  0.000000  1.0  1.000000  0.018862  0.004415   
            21 4  1.000000  0.000000  1.0  1.000000  0.001141  0.000128   
               5  1.000000  0.000000  1.0  1.000000  0.004191  0.000273   
               6  1.000000  0.000000  1.0  1.000000  0.018051  0.001144   
pruned      11 4  1.000631  0.008174  1.0  1.166667  0.060281  0.013391   
               5  1.000017  0.001071  1.0  1.066667  0.101555  0.027070   
               6  1.000055  0.003898  1.0  1.375000  0.226075  0.122718   
            21 4  1.000296  0.003716  1.0  1.062500  0.139488  0.034453   
               5  1.000007  0.000459  1.0  1.028571  0.290511  0.121911   
               6  1.000678  0.011957  1.0  1.500000  0.869537  0.478293   

                                      
                       min       max  
alg         m  n                      
brute force 11 4  0.000915  0.001508  
               5  0.003718  0.008938  
               6  0.016475  0.101547  
            21 4  0.000997  0.002764  
               5  0.003819  0.008590  
               6  0.016832  0.062779  
pruned      11 4  0.048071  0.102799  
               5  0.066841  0.343291  
               6  0.090097  2.207894  
            21 4  0.100150  0.310969  
               5  0.146970  1.284809  
               6  0.205735  2.402603

In [8]:
def generate_prior_grid(n_components, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls - n_components):
        counts = np.bincount(combo, minlength=n_components) + 1
        grids.append((counts * step).round(4))
    return grids

In [9]:
df_milp_over_1 = df_res_ratio[(df_res_ratio["alg"]=="pruned") & (df_res_ratio["ratio"]> 1+1e-12)]
print(f"{len(df_milp_over_1)} examples where solver and OPT mismatch")

75 examples where solver and OPT mismatch


In [10]:
df_milp_over_1

,m,n,i,alg,ratio,time
969,11,4,0,pruned,1.055556,0.056114
970,11,4,1,pruned,1.055556,0.055469
977,11,4,8,pruned,1.066667,0.052862
978,11,4,9,pruned,1.066667,0.050286
986,11,4,17,pruned,1.066667,0.055120
...,...,...,...,...,...,...
63366,21,6,9102,pruned,1.125000,2.209345
64648,21,6,10384,pruned,1.250000,0.615703
65137,21,6,10873,pruned,1.400000,2.140914
65241,21,6,10977,pruned,1.066667,1.053373


In [11]:
row = df_milp_over_1.iloc[df_milp_over_1["ratio"].argmax()]
m, n, i = row.m, row.n, row.i
df_cp[(df_cp["m"]==m) & (df_cp["n"]==n) & (df_cp["i"]==i)]

,i,alg,time,loss,partition,m,n
52638,5001,brute force,0.017796,0.042857,"[[1], [0, 2, 3, 4], [5]]",21,6
52639,5001,pruned,2.210009,0.064286,"[[0, 1], [2, 3, 4, 5]]",21,6


In [12]:
pg = generate_prior_grid(n, 20)
priors = pg[i]
thresholds = np.linspace(0., 1., n)
tt = 0.1
c = 1.0

mip = solve(priors, thresholds, tt, c, m)
# mip = solve_milp(priors, thresholds, tt, c, m, solver=cp.GUROBI)
mip["loss"], mip["partition"]

(0.04285714285714286, [[0, 2, 3, 4], [1], [5]])

In [13]:
df_cp[(df_cp["m"]==m) & (df_cp["n"]==n) & (df_cp["i"]==i)].iloc[1].loss

np.float64(0.06428571428571428)

In [27]:
results_over_1 = {"i": [], "loss": [], "ratio": [], "partition": []}
for _, row in df_milp_over_1.iterrows():
    m, n, i = row.m, row.n, row.i
    pg = generate_prior_grid(n, 20)
    priors = pg[i]
    thresholds = np.linspace(0., 1., n)
    tt = 0.1
    c = 1.0

    mip = solve_milp(priors, thresholds, tt, c, m)
    p_mip, loss_mip = mip["partition"], mip["loss"]
    r_mip = loss_mip / df_cp[(df_cp["m"]==m) & (df_cp["n"]==n) & (df_cp["i"]==i)].iloc[0].loss
    
    results_over_1["i"].append(i)
    results_over_1["loss"].append(loss_mip)
    results_over_1["ratio"].append(r_mip)
    results_over_1["partition"].append(p_mip)

In [34]:
df_results_over_1 = pd.DataFrame(results_over_1)
df_fails = df_results_over_1[df_results_over_1["ratio"] > 1+1e-12]
print(f"Even after removing time limit {len(df_fails)} examples weren't solved")

Even after removing time limit 0 examples weren't solved


In [35]:
df_fails

,i,loss,ratio,partition
